In [5]:
import numpy as np
import torch
from torch import nn
import pandas as pd
import os
from reetoolbox.dataloaders import *
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split

PATH = "../.."
# PATH = "storage"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Data Setup

In [6]:
# from torchvision import transforms
# from torch.utils.data import DataLoader, Subset
# from sklearn.model_selection import train_test_split

# # Parameters
# dataset_name = "PanNuke"
# root_dir = os.path.join(PATH, 'data', dataset_name)
# batch_size = 8
# test_multiplier = 0.28
# # test_multiplier = 0.01

# # Transforms
# transform = transforms.Compose([
#     transforms.ToTensor(),
# ])

# # For test set (Fold 3 only, with stratified subsample)
# test_ds = PanNukeDataset(
#     root_dir, folds=(3,), transform=transform, min_positive=5
# )

# # Stratified subsample for test set
# ix_test = list(range(len(test_ds)))
# labels_test = [test_ds.samples[i]['label'] for i in ix_test]

# test_ix, _ = train_test_split(
#     ix_test,
#     train_size=test_multiplier,       # e.g., 0.25 for a 25% stratified subsample
#     stratify=labels_test,
#     random_state=42
# )
# test_data = Subset(test_ds, test_ix)
# test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

# print(f"Test batches: {len(test_loader)}")

In [7]:
# dataset_name = "NCT"
# batch_size           = 8
# mode                 = 'tum_vs_all'
# test_multiplier      = 0.012
# test_root            = os.path.join(PATH, 'data', 'NCT', 'CRC-VAL-HE-7K')
# mode                 = 'tum_vs_all'

# test_data, test_loader = build_dataset(
#     NCTDataSet,
#     root_dir=test_root,
#     batch_size=batch_size,
#     test_multiplier=test_multiplier,
#     transform=transforms.Compose([transforms.ToTensor()]),
#     classification_mode=mode,
#     shuffle=False
# )

# print(f"Test batches: {len(test_loader)}")

In [8]:
# dataset_name = "PanNuke"
# root_dir = os.path.join(PATH, 'data', dataset_name)
# batch_size = 8
# test_multiplier = 0.035

# test_data, test_loader = build_dataset(
#     PanNukeDataset,
#     root_dir=root_dir,
#     batch_size=batch_size,
#     test_multiplier=test_multiplier,
#     transform=transforms.Compose([transforms.ToTensor()]),
#     folds=(3,),
#     min_positive=5,
#     shuffle=False
# )

# print(f"Test batches: {len(test_loader)}")

In [9]:
# dataset_name = "PANDA"

# panda_root = os.path.join(PATH, 'data', dataset_name)
# batch_size = 8
# test_multipler = 0.012

# test_data, test_loader = build_dataset(
#     PandaDataset, 
#     root_dir = panda_root, 
#     batch_size = batch_size, 
#     test_multiplier = test_multipler, 
#     transform=transforms.Compose([transforms.ToTensor()]), 
#     split='test'
# )

# print(f"Test batches: {len(test_loader)}")

In [16]:
dataset_name = "PatchCamelyon"

patch_root    = os.path.join(PATH, 'data', 'PatchCamelyon')
batch_size    = 8
test_multiplier = 0.022

test_data, test_loader = build_dataset(
    PatchCamelyonDataset,
    root_dir = patch_root,
    batch_size = batch_size,
    test_multiplier = test_multiplier,
    transform = transforms.Compose([
        transforms.ToTensor()
    ]),
    split = 'test'
)

print(f"Test batches: {len(test_loader)}")

Test batches: 90


In [11]:
counts = torch.zeros(2, dtype=torch.long)
for _, labels, _ in test_loader:
    counts += torch.bincount(labels, minlength=2)

print(counts)

tensor([43, 43])


In [ ]:
import os
import sys
from huggingface_hub import login, hf_hub_download
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from timm.layers import SwiGLUPacked
from transformers import AutoModel, AutoImageProcessor
github_path = os.path.join(PATH, "third_party", "EXAONEPath")
sys.path.append(github_path)
from vision_transformer import VisionTransformer

# — login once per session —
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)

def _attach_head_and_freeze(model: nn.Module, n_classes: int, device: str):
    # infer feature‐dim
    num_features = getattr(model, 'num_features', None)
    if num_features is None:
        # fallback for timm classifiers named .head or .classifier
        if hasattr(model, 'head') and hasattr(model.head, 'in_features'):
            num_features = model.head.in_features
        elif hasattr(model, 'classifier') and hasattr(model.classifier, 'in_features'):
            num_features = model.classifier.in_features
        else:
            raise AttributeError("Cannot infer feature dimension")
    # replace head
    model.head = nn.Sequential(
        nn.Linear(num_features, n_classes),
        nn.LogSoftmax(dim=1)
    )
    # freeze backbone
    for p in model.parameters():
        p.requires_grad = False
    # unfreeze head
    for p in model.head.parameters():
        p.requires_grad = True
    model = model.to(device).train()
    optim_head = optim.SGD(model.head.parameters(), lr=1e-3, momentum=0.9)
    return model, optim_head

# ---------------------------------------------------------------------
# 2) helpers for HF vision models (they don’t expose .head)
# ---------------------------------------------------------------------
class HFCLSWrapper(nn.Module):
    """
    Wrap a HuggingFace vision backbone so that:
      • forward() returns CLS features
      • .num_features is available for your helper
      • .head exists (will be replaced)
    """
    def __init__(self, backbone: AutoModel):
        super().__init__()
        self.backbone = backbone
        self.num_features = getattr(
            backbone.config, "hidden_size",
            getattr(backbone.config, "vision_dim", None)
        )
        if self.num_features is None:
            raise ValueError("Cannot find hidden size in config")
        self.head = nn.Identity()      # placeholder – replaced later

    def forward(self, pixel_values=None, **kwargs):
        outputs = self.backbone(pixel_values=pixel_values, **kwargs)
        # outputs.last_hidden_state → (B, N+1, D); CLS is at index 0
        last_hidden = (
            outputs.last_hidden_state
            if hasattr(outputs, "last_hidden_state")
            else outputs[0]
        )
        feats = last_hidden[:, 0, :]           # (B, D)
        return self.head(feats)

# ---------------------------------------------------------------------
# 3) one-liner loaders
# ---------------------------------------------------------------------
def load_h_optimus(n_classes: int = 2, device: str = "cuda"):
    """bioptimus/H-optimus-0 (ViT-B/16)"""
    timm_kwargs = dict(init_values=1e-5, dynamic_img_size=False)
    model = timm.create_model(
        "hf-hub:bioptimus/H-optimus-0",
        pretrained=True, num_classes=0, **timm_kwargs
    )
    return _attach_head_and_freeze(model, n_classes, device)


def load_exaonepath(n_classes: int = 2, device: str = "cuda"):
    """EXAONEPath loader that properly wires backbone and head"""

    backbone = VisionTransformer.from_pretrained("LGAI-EXAONE/EXAONEPath")

    # Wrapper Module
    class ExaoneWrapper(nn.Module):
        def __init__(self, backbone, n_classes):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(
                nn.Linear(backbone.embed_dim, n_classes),
                nn.LogSoftmax(dim=1)
            )

            # Freeze backbone
            for p in self.backbone.parameters():
                p.requires_grad = False

        def forward(self, x):
            feats = self.backbone(x)  # <-- ensure this returns raw features, no .detach()
            return self.head(feats)

    model = ExaoneWrapper(backbone, n_classes).to(device).train()
    optimizer = optim.SGD(model.head.parameters(), lr=1e-3, momentum=0.9)
    return model, optimizer

def load_hibou_b(n_classes: int = 2, device: str = "cuda"):
    """histai/hibou-b"""
    processor = AutoImageProcessor.from_pretrained("histai/hibou-b", trust_remote_code=True)
    backbone  = AutoModel.from_pretrained("histai/hibou-b", trust_remote_code=True)
    model, opt = _attach_head_and_freeze(HFCLSWrapper(backbone), n_classes, device)
    return processor, model, opt

def load_hibou_l(n_classes: int = 2, device: str = "cuda"):
    """histai/hibou-L"""
    processor = AutoImageProcessor.from_pretrained("histai/hibou-L", trust_remote_code=True)
    backbone  = AutoModel.from_pretrained("histai/hibou-L", trust_remote_code=True)
    model, opt = _attach_head_and_freeze(HFCLSWrapper(backbone), n_classes, device)
    return processor, model, opt

def _load_phikon_like(repo_id: str, n_classes: int, device: str):
    processor = AutoImageProcessor.from_pretrained(repo_id)
    backbone  = AutoModel.from_pretrained(repo_id)
    model, opt = _attach_head_and_freeze(HFCLSWrapper(backbone), n_classes, device)
    return processor, model, opt

def load_phikon(n_classes: int = 2, device: str = "cuda"):
    """owkin/phikon (v1)"""
    return _load_phikon_like("owkin/phikon", n_classes, device)

def load_phikon_v2(n_classes: int = 2, device: str = "cuda"):
    """owkin/phikon-v2"""
    return _load_phikon_like("owkin/phikon-v2", n_classes, device)

def load_uni2(n_classes: int = 2, device: str = "cuda:0"):
    """MahmoodLab/UNI2-h (ViT‑H/14)"""
    timm_kwargs = {
        'img_size': 224, 'patch_size': 14, 'depth': 24, 'num_heads': 24,
        'init_values': 1e-5, 'embed_dim': 1536, 'mlp_ratio': 2.66667*2,
        'num_classes': 0, 'no_embed_class': True, 'mlp_layer': timm.layers.SwiGLUPacked,
        'act_layer': nn.SiLU, 'reg_tokens': 8, 'dynamic_img_size': True
    }
    model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)
    return _attach_head_and_freeze(model, n_classes, device)

def load_uni(n_classes: int = 2, device: str = "cuda:0"):
    """MahmoodLab/UNI (ViT‑L/16) — must pass init_values & dynamic_img_size"""
    timm_kwargs = {
        'init_values': 1e-5,
        'dynamic_img_size': True
    }
    # note repo id is lowercase "uni"
    model = timm.create_model(
        "hf-hub:MahmoodLab/uni",
        pretrained=True,
        num_classes=0,
        **timm_kwargs
    )
    return _attach_head_and_freeze(model, n_classes, device)

def load_gigapath(n_classes: int = 2, device: str = "cuda:0"):
    """prov‑gigapath/prov‑gigapath patch‑level model with a binary head."""
    model = timm.create_model(
        "hf-hub:prov-gigapath/prov-gigapath",
        pretrained=True,
        num_classes=0
    )
    return _attach_head_and_freeze(model, n_classes, device)

def load_resnet18(n_classes: int = 2, device: str = "cuda:0"):
    model = models.resnet18(pretrained=True)  
    model.fc = nn.Sequential(nn.Linear(512, n_classes),        
                            nn.LogSoftmax(dim=1))
    model = model.to(device)
    model.train()  

    params_to_update = model.parameters()
    optimizer_ft = optim.SGD(params_to_update, lr=0.001, momentum=0.9)
    return model, optimizer_ft

def load_resnet50(n_classes: int = 2, device: str = "cuda:0"):
    model = models.resnet50(pretrained=True)  
    model.fc = nn.Sequential(nn.Linear(2048, n_classes),        
                            nn.LogSoftmax(dim=1))
    model = model.to(device)
    model.train()  

    params_to_update = model.parameters()
    optimizer_ft = optim.SGD(params_to_update, lr=0.001, momentum=0.9)
    return model, optimizer_ft

# ─── Virchow loaders ───────────────────────────────────────────────
def load_virchow(n_classes: int = 2, device: str = "cuda"):
    """
    paige-ai/Virchow — ViT-L with 4 register tokens.
    2 560-d feature vector: [CLS token, mean(patch tokens)].
    Returns (model_with_head, optimizer_for_head).
    """
    import timm
    import torch
    import torch.nn as nn
    from timm.layers import SwiGLUPacked

    vit = timm.create_model(
        "hf-hub:paige-ai/Virchow",
        pretrained=True,
        mlp_layer=SwiGLUPacked,
        act_layer=torch.nn.SiLU,
        num_classes=0
    )

    class VirchowWrapper(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            self.num_features = backbone.embed_dim * 2
            self.head = nn.Identity()

        def forward(self, x):
            tokens = self.backbone(x)
            cls_tok = tokens[:, 0]
            patch_tok = tokens[:, 5:].mean(1)
            feats = torch.cat([cls_tok, patch_tok], dim=-1)
            return self.head(feats)

    wrapper = VirchowWrapper(vit)
    return _attach_head_and_freeze(wrapper, n_classes, device)

def load_virchow2(n_classes: int = 2, device: str = "cuda"):
    """
    paige-ai/Virchow2 — ViT-L with 4 register tokens.
    A 2 560-d feature vector is built from:
        [CLS token,  mean(patch tokens)].
    Returns (model_with_head, optimizer_for_head)
    """
    import timm
    import torch
    import torch.nn as nn
    from timm.layers import SwiGLUPacked

    # 1. backbone (returns all tokens, no classifier head)
    vit = timm.create_model(
        "hf-hub:paige-ai/Virchow2",
        pretrained=True,
        mlp_layer=SwiGLUPacked,          # required by repo
        act_layer=torch.nn.SiLU,
        num_classes=0                    # we want features, not logits
    )

    # 2. Wrap to expose a single feature tensor + num_features
    class Virchow2Wrapper(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            self.num_features = backbone.embed_dim * 2        # 1 280 × 2 = 2 560
            self.head = nn.Identity()                         # will be replaced

        def forward(self, x):
            tokens = self.backbone(x)            # (B, 261, 1280)
            cls_tok = tokens[:, 0]               # (B, 1280)
            patch_tok = tokens[:, 5:].mean(1)    # ignore 4 register tokens
            feats = torch.cat([cls_tok, patch_tok], dim=-1)   # (B, 2560)
            return self.head(feats)

    wrapper = Virchow2Wrapper(vit)

    # 3. Add your classification head & freeze backbone
    return _attach_head_and_freeze(wrapper, n_classes, device)

### Evaluation

In [ ]:
from reetoolbox.optimisers import PGD, StochasticSearch
from reetoolbox.image_evaluator import Evaluator
from reetoolbox.metrics import get_metrics

from reetoolbox.transforms import (
    PixelTransform,          # → eval_pixel_optimiser_params, eval_pixel_transform_params
    StainTransform,          # → eval_stain_optimiser_params, eval_stain_transform_params
    MeanTransform,           # → eval_mean_optimiser_params, eval_mean_transform_params
    RotateTransform,         # → eval_rotate_optimiser_params, eval_rotate_transform_params
    CropTransform,           # → eval_crop_optimiser_params, eval_crop_transform_params
    BlurTransform,           # → eval_blur_optimiser_params, eval_blur_transform_params
    ZoomInTransform,         # → eval_zoom_in_optimiser_params, eval_zoom_in_transform_params
    ZoomOutTransform,        # → eval_zoom_out_optimiser_params, eval_zoom_out_transform_params
    HEDTransform,            # → eval_hed_optimiser_params, eval_hed_transform_params
    RandomStainTransform,    # → eval_random_stain_optimiser_params, eval_random_stain_transform_params
    JPEGTransform,           # → eval_jpeg_optimiser_params, eval_jpeg_transform_params
)

from reetoolbox.constants import (
    eval_pixel_optimiser_params,        eval_pixel_transform_params,
    eval_stain_optimiser_params,        eval_stain_transform_params,
    eval_mean_optimiser_params,         eval_mean_transform_params,
    eval_crop_optimiser_params,         eval_crop_transform_params,
    eval_rotate_optimiser_params,       eval_rotate_transform_params,
    eval_crop_optimiser_params,         eval_crop_transform_params,
    eval_blur_optimiser_params,         eval_blur_transform_params,
    eval_zoom_in_optimiser_params,      eval_zoom_in_transform_params,
    eval_zoom_out_optimiser_params,     eval_zoom_out_transform_params,
    eval_hed_optimiser_params,          eval_hed_transform_params,
    eval_random_stain_optimiser_params, eval_random_stain_transform_params,
    eval_jpeg_optimiser_params,         eval_jpeg_transform_params,
)

def evaluate_pixel(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "pixel"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        PGD, PixelTransform,
        eval_pixel_optimiser_params, eval_pixel_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_stain(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "stain"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        PGD, StainTransform,
        eval_stain_optimiser_params, eval_stain_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_brightness(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "brightness"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        PGD, MeanTransform,
        eval_mean_optimiser_params, eval_mean_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_rotate(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "rotate"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, RotateTransform,
        eval_rotate_optimiser_params, eval_rotate_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_crop(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "crop"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, CropTransform,
        eval_crop_optimiser_params, eval_crop_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_blur(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "blur"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, BlurTransform,
        eval_blur_optimiser_params, eval_blur_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_zoomin(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "zoomin"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, ZoomInTransform,
        eval_zoom_in_optimiser_params, eval_zoom_in_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_zoomout(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "zoomout"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, ZoomOutTransform,
        eval_zoom_out_optimiser_params, eval_zoom_out_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_hed(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "hed"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, HEDTransform,
        eval_hed_optimiser_params, eval_hed_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_random_stain(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "random_stain"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, RandomStainTransform,
        eval_random_stain_optimiser_params, eval_random_stain_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))

def evaluate_jpeg(model, model_name, test_dataset, test_dataloader, save_dir_base):
    perturb = "jpeg"
    save_dir = os.path.join(save_dir_base, model_name, perturb)
    evaluator = Evaluator(
        model, model_name, 
        test_dataset, test_dataloader,
        StochasticSearch, JPEGTransform,
        eval_jpeg_optimiser_params, eval_jpeg_transform_params,
        device=device, save_dir=save_dir
    )
    return get_metrics(evaluator.predict(adversarial=True))


In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params} ({total_params/1e6:.2f}M)")
    return total_params

# ─── collect_model_stats with save_dir_base argument ─────────────────────────
def collect_model_stats(
    model,
    model_name: str,
    test_data,
    test_loader,
    save_dir_base: str,
    runs: int = 5,
    perturbations: list = None,
    metric_names: list = None
) -> pd.DataFrame:
    import numpy as np
    import pandas as pd
    import copy

    # Define metric renaming
    metric_rename = {
        "Accuracy": "Baseline Accuracy",
        "ROC AUC": "Baseline ROC",
        "PR AUC": "Baseline PR",
        "Robustness Accuracy": "Accuracy After Perturbation"
    }

    metric_names_full = [
        "Accuracy", "ROC AUC", "PR AUC",
        "Robustness Accuracy", "Fooling Ratio", "Input Sensitivity",
        "Adversarial ROC AUC", "Adversarial PR AUC"
    ]
    mean_only_metrics = ["Accuracy", "ROC AUC", "PR AUC"]
    std_metrics = [m for m in metric_names_full if m not in mean_only_metrics]

    if perturbations is None:
        perturbations = [
            "pixel", "stain", "brightness", "rotate", "crop",
            "blur", "zoomin", "zoomout", "hed", "random_stain", "jpeg"
        ]
    if metric_names is None:
        metric_names = metric_names_full

    # Build MultiIndex columns with renamed metrics
    col_tuples = []
    for m in metric_names:
        m_renamed = metric_rename.get(m, m)
        col_tuples.append((model_name, m_renamed, "Mean"))
        if m in std_metrics:
            col_tuples.append((model_name, m_renamed, "STD"))
    columns = pd.MultiIndex.from_tuples(
        col_tuples, names=["Model", "Metric", "Statistic"]
    )

    # DataFrame indexed by perturbation name
    stats_df = pd.DataFrame(
        index=perturbations,
        columns=columns,
        dtype=float
    )

    for i, p in enumerate(perturbations):
        print(f"-------- {i+1}/{len(perturbations)}: Evaluating '{p}' --------")
        # {metric: [run_1, run_2, ...]}
        raw = {m: [] for m in metric_names_full}

        eval_fn = globals()[f"evaluate_{p}"]
        for run_idx in range(runs):
            print(f"  Run {run_idx+1}/{runs}...", end="")
            vals = eval_fn(
                model,
                model_name,
                test_data,
                test_loader,
                save_dir_base
            )
            print(" done")
            # Now vals = (acc, roc, pr, robust_acc, fool_ratio, input_sens, ad_roc, ad_pr)
            for m, v in zip(metric_names_full, vals):
                raw[m].append(v)

        # fill means & STDs
        for m in metric_names:
            m_renamed = metric_rename.get(m, m)
            stats_df.loc[p, (model_name, m_renamed, "Mean")] = np.mean(raw[m])
            if m in std_metrics:
                stats_df.loc[p, (model_name, m_renamed, "STD")] = np.std(raw[m])

    # Compute and append the "Mean" row (averaged across perturbations)
    mean_row = {}
    for m in metric_names:
        m_renamed = metric_rename.get(m, m)
        mean_row[(model_name, m_renamed, "Mean")] = stats_df.loc[:, (model_name, m_renamed, "Mean")].mean()
        if m in std_metrics:
            mean_row[(model_name, m_renamed, "STD")] = stats_df.loc[:, (model_name, m_renamed, "STD")].mean()

    stats_df.loc["Mean"] = pd.Series(mean_row)
    stats_df = stats_df.reindex(perturbations + ["Mean"])

    return stats_df

In [ ]:
model_evals = [
    # {
    #     "model_name": "ResNet18",
    #     "load_func": lambda: torch.load(os.path.join(PATH, "Models", dataset_name, "resnet18_pannuke.pth"), weights_only=False),
    #     "weight_path": None,  # already loaded with torch.load
    #     "test_loader_key": "resnet",
    #     "output_subdir": "ResNet18",
    # },
    # {
    #     "model_name": "ResNet50",
    #     "load_func": lambda: torch.load(os.path.join(PATH, "Models", dataset_name, "resnet50_pannuke.pth"), weights_only=False),
    #     "weight_path": None,  # already loaded with torch.load
    #     "test_loader_key": "resnet",
    #     "output_subdir": "ResNet50",
    # },
    # {
    #     "model_name": "UNI",
    #     "load_func": lambda: load_uni(n_classes=2)[0],
    #     "weight_path": os.path.join(PATH, "Models", dataset_name, "uni_pannuke.pth"),
    #     "test_loader_key": "UNI",
    #     "output_subdir": "UNI",
    # },
    # {
    #     "model_name": "UNI2",
    #     "load_func": lambda: load_uni2(n_classes=2)[0],
    #     "weight_path": os.path.join(PATH, "Models", dataset_name, "uni2_pannuke.pth"),
    #     "test_loader_key": "UNI2",
    #     "output_subdir": "UNI2",
    # },
    # {
    #     "model_name": "GigaPath",
    #     "load_func": lambda: load_gigapath(n_classes=2)[0],
    #     "weight_path": os.path.join(PATH, "Models", dataset_name, "gigapath_pannuke.pth"),
    #     "test_loader_key": "GigaPath",
    #     "output_subdir": "GigaPath",
    # },
    # {
    #     "model_name": "Virchow",
    #     "load_func": lambda: load_virchow(n_classes=2, device=device)[0],
    #     "weight_path": os.path.join(PATH, "Models", dataset_name, "virchow_pannuke.pth"),
    #     "test_loader_key": "Virchow",
    #     "output_subdir": "Virchow",
    # },
    # {
    #     "model_name": "Virchow2",
    #     "load_func": lambda: load_virchow2(n_classes=2, device=device)[0],
    #     "weight_path": os.path.join(PATH, "Models", dataset_name, "virchow2_pannuke.pth"),
    #     "test_loader_key": "Virchow2",
    #     "output_subdir": "Virchow2",
    # },
    {
        "model_name": "H-Optimus-0",
        "load_func": lambda: load_h_optimus(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, "hoptimus0_pannuke.pth"),
        "test_loader_key": "H-optimus-0",
        "output_subdir": "H-Optimus-0",
    },
    {
        "model_name": "EXAONEPath",
        "load_func": lambda: load_exaonepath(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, "exaonepath_pannuke.pth"),
        "test_loader_key": "EXAONEPath",
        "output_subdir": "EXAONEPath",
    },
    {
        "model_name": "Hibou-B",
        "load_func": lambda: load_hibou_b(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, "hibou_b_pannuke.pth"),
        "test_loader_key": "Hibou-B",
        "output_subdir": "Hibou-B",
    },
    {
        "model_name": "Hibou-L",
        "load_func": lambda: load_hibou_l(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, "hibou_l_pannuke.pth"),
        "test_loader_key": "Hibou-L",
        "output_subdir": "Hibou-L",
    },
    {
        "model_name": "Phikon v1",
        "load_func": lambda: load_phikon(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, "phikon_pannuke.pth"),
        "test_loader_key": "Phikon v1",
        "output_subdir": "Phikon v1",
    },
    {
        "model_name": "Phikon v2",
        "load_func": lambda: load_phikon_v2(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, "phikon_v2_pannuke.pth"),
        "test_loader_key": "Phikon v2",
        "output_subdir": "Phikon v2",
    },
]

In [ ]:
# Cell 1: ProcessorTransform utility for HuggingFace processors
class ProcessorTransform:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, img):
        # Accepts a PIL image; handles resizing, normalization, etc.
        return self.processor(img, return_tensors="pt")["pixel_values"].squeeze(0)


# Cell 2: Standard ImageNet transform for torchvision/timm models
from torchvision import transforms

imagenet_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# Cell 3: H-Optimus-specific normalization
hoptimus_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517))
])

# Cell 4: Instantiate HuggingFace processors and wrap as transforms
from transformers import AutoImageProcessor

hibou_b_processor = AutoImageProcessor.from_pretrained("histai/hibou-b", trust_remote_code=True)
hibou_L_processor = AutoImageProcessor.from_pretrained("histai/hibou-L", trust_remote_code=True)
phikon_processor = AutoImageProcessor.from_pretrained("owkin/phikon", trust_remote_code=True)
phikonv2_processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2", trust_remote_code=True)
resnet18_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
resnet50_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")

hibou_b_transform  = ProcessorTransform(hibou_b_processor)
hibou_L_transform  = ProcessorTransform(hibou_L_processor)
phikon_transform   = ProcessorTransform(phikon_processor)
phikonv2_transform = ProcessorTransform(phikonv2_processor)
resnet18_transform = ProcessorTransform(resnet18_processor)
resnet50_transform = ProcessorTransform(resnet50_processor)

# Cell 5: For timm-based models (e.g. Virchow), builds official transform
from timm.data import resolve_data_config, create_transform

def timm_transform(model):
    # Try to access pretrained_cfg at the top level, or in a .backbone attribute
    if hasattr(model, "pretrained_cfg"):
        timm_model = model
    elif hasattr(model, "backbone") and hasattr(model.backbone, "pretrained_cfg"):
        timm_model = model.backbone
    else:
        raise AttributeError("Cannot find .pretrained_cfg on the model or its backbone.")
    cfg = resolve_data_config(timm_model.pretrained_cfg, model=timm_model)
    return create_transform(**cfg)


# Cell 6: Model-to-transform mapping
model_transforms = {
    "ResNet18": resnet18_transform,       # If using HuggingFace, else imagenet_transform
    "ResNet50": resnet50_transform,       # If using HuggingFace, else imagenet_transform
    "UNI": imagenet_transform,
    "UNI2": imagenet_transform,
    "GigaPath": imagenet_transform,
    "EXAONEPath": imagenet_transform,
    "H-Optimus-0": hoptimus_transform,
    "H-Optimus-1": hoptimus_transform,
    "Hibou-B": hibou_b_transform,
    "Hibou-L": hibou_L_transform,
    "Phikon v1": phikon_transform,
    "Phikon v2": phikonv2_transform,
    "Virchow": None,     # Use timm_transform in get_transform for these
    "Virchow2": None,
    # "H0-mini": None,   # Add for other timm-based models as needed
}

# Cell 7: get_transform that handles timm-based special cases
def get_transform(model_name, model=None):
    if model_name in {"Virchow", "Virchow2", "H0-mini"} and model is not None:
        return timm_transform(model)
    return model_transforms[model_name]


In [ ]:
runs = 3  # number of runs for each perturbation

In [ ]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from torchvision import transforms
import os

def run_all_model_stats(
    model_evals,
    root_dir,
    batch_size,
    test_multiplier,
    runs,
    device,
    PATH,
    save_dir_base,    # e.g. os.path.join(PATH, "Stats", "PanNuke")
    dataset_class=PanNukePatchDataset,
    dataset_kwargs={'folds': (3,), 'min_positive': 5}
):
    # 1. Get full label list from a dummy dataset (for stratification)
    dummy_transform = transforms.Compose([transforms.ToTensor()])
    dummy_ds = dataset_class(root_dir, transform=dummy_transform, **dataset_kwargs)
    ix_test = list(range(len(dummy_ds)))
    if hasattr(dummy_ds, 'labels'):
        labels_test = [dummy_ds.labels[i] for i in ix_test]
    elif hasattr(dummy_ds, 'samples'):
        labels_test = [dummy_ds.samples[i]['label'] for i in ix_test]
    else:
        raise AttributeError("Dataset must have .labels or .samples for stratification.")

    # 2. Stratified subsample to get test_ix
    test_ix, _ = train_test_split(
        ix_test,
        train_size=test_multiplier,
        stratify=labels_test,
        random_state=42,
    )

    # 3. Loop over each model, load and evaluate
    for model_eval in model_evals:
        print(f"\n----------------------Beginning {model_eval['model_name']} stats evaluation----------------------")
        
        # a. Load the model
        model = model_eval["load_func"]()
        if model_eval["weight_path"] is not None:
            model.head.load_state_dict(torch.load(model_eval["weight_path"]))
        model = model.to(device).eval()

        # b. Get the correct transform for this model
        transform = get_transform(model_eval["model_name"], model=model)

        # c. Build the dataset and DataLoader using only the test_ix indices
        test_full = dataset_class(root_dir, transform=transform, **dataset_kwargs)
        test_data = Subset(test_full, test_ix)
        test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

        # d. Collect stats
        stats_df = collect_model_stats(
            model=model,
            model_name=model_eval["model_name"],
            test_data=test_data,
            test_loader=test_loader,
            save_dir_base=save_dir_base,
            runs=runs
        )

        # e. Save to Excel
        save_loc = os.path.join(
            save_dir_base, model_eval["output_subdir"], f"{model_eval['model_name']}_stats.xlsx"
        )
        os.makedirs(os.path.dirname(save_loc), exist_ok=True)
        stats_df.to_excel(save_loc)
        print(f"Saved stats for {model_eval['model_name']} to {save_loc}")


In [ ]:
run_all_model_stats(
    model_evals=model_evals,
    root_dir=root_dir,
    batch_size=8,
    test_multiplier=test_multiplier,
    runs=runs,
    device=device,
    PATH=PATH,
    save_dir_base=os.path.join(PATH, "Stats", "PanNuke"),
    dataset_class=PanNukePatchDataset,
    dataset_kwargs={'folds': (3,), 'min_positive': 5}
)

#### Compile stats csv

In [ ]:
# resnet_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "ResNet", "ResNet_stats.xlsx"))
# uni_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "UNI", "UNI_stats.xlsx"))
# uni2_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "UNI2", "UNI2_stats.xlsx"))
# gigapath_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "GigaPath", "GigaPath_stats.xlsx"))
# virchow_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "Virchow", "Virchow_stats.xlsx"))
# virchow2_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "Virchow2", "Virchow2_stats.xlsx"))
# hoptimus_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "H-Optimus-0", "H-Optimus-0_stats.xlsx"))
# exaonepath_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "EXAONEPath", "EXAONEPath_stats.xlsx"))
# hiboub_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "Hibou-B", "Hibou-B_stats.xlsx"))
# phikonv1_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "Phikon v1", "Phikon v1_stats.xlsx"))
# phikonv2_stats_df = pd.read_excel(os.path.join(PATH, "Stats", "PanNuke", "Phikon v2", "Phikon v2_stats.xlsx"))

In [ ]:
# all_stats = {
#     "ResNet": resnet_stats_df,
#     "UNI": uni_stats_df,
#     "UNI2": uni2_stats_df,
#     "GigaPath": gigapath_stats_df,
#     "Virchow": virchow_stats_df,
#     "Virchow2": virchow2_stats_df,
#     "H-Optimus-0": hoptimus_stats_df,
#     "EXAONEPath": exaonepath_stats_df,
#     "Hibou-B": hiboub_stats_df,
#     "Phikon v1": phikonv1_stats_df,
#     "Phikon v2": phikonv2_stats_df
# }

# output_filepath = os.path.join(PATH, "Stats", "PanNuke", "all_models_perturbation_stats.xlsx")

# with pd.ExcelWriter(output_filepath, engine="openpyxl") as writer:
#     for model_name, stats_df in all_stats.items():
#         # Write each DataFrame to a sheet named after the model
#         stats_df.to_excel(writer, sheet_name=model_name, index=True)

# print(f"Saved combined stats to: {output_filepath}")